In [ ]:
import cv2, numpy as np

cap = cv2.VideoCapture('Robots.mp4')  #opens the video file so we can read frames in order.
ret, prev = cap.read()  #cap.read() grabs the first frame of the video (prev).
prev_gray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)  #Convert that frame to grayscale (prev_gray), because optical flow works on single-channel images.

step = 5   # spacing between arrows

while True:  #Loop until there are no more frames.
    ret, frame = cap.read()
    if not ret: break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Dense optical flow
    flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None,                
                                        0.5, 3, 15, 3, 5, 1.5, 0)   #estimates motion for every pixel between the previous and current grayscale frames.
    mag, ang = cv2.cartToPolar(flow[...,0], flow[...,1])  #cv2.cartToPolar converts these x/y displacements into: mag = magnitude (speed of movement), ang = angle (direction of movement)

    # Draw arrows on a grid
    h, w = gray.shape
    for y in range(0, h, step):
        for x in range(0, w, step):
            if mag[y, x] > 1:  # draw only if motion is significant
                fx, fy = flow[y, x]
                cv2.arrowedLine(frame,
                                 (x, y),
                                 (int(x + fx), int(y + fy)),
                                 (0, 255, 0), 1, tipLength=0.4)

    cv2.imshow('Dense Flow Arrows', frame)
    if cv2.waitKey(20) & 0xFF in (27, ord('q')): break

    prev_gray = gray

cap.release()
cv2.destroyAllWindows()
